In [1]:
import zipfile
from pathlib import Path
import pandas as pd

raw_dir = Path("/home/eprashar_solutions_corelogic_com/network-idx/data/raw/census/baf2020")
extract_dir = Path("/home/eprashar_solutions_corelogic_com/network-idx/data/extracted/census/baf2020")
extract_dir.mkdir(parents=True, exist_ok=True)

In [3]:
# Grab the first zip file in the directory
zip_file = next(raw_dir.glob("*.zip"))
print(f"Extracting: {zip_file.name}")

with zipfile.ZipFile(zip_file, "r") as zf:
    print("Contents:", zf.namelist())
    txt_name = [n for n in zf.namelist() if "INCPLACE_CDP" in n][0]
    zf.extract(txt_name, path=extract_dir)

txt_path = extract_dir / txt_name
df = pd.read_csv(txt_path, sep="|", dtype=str)
print(df.info())
df.head()

Extracting: BlockAssign_ST01_AL_2020.zip
Contents: ['BlockAssign_ST01_AL_AIANNH.txt', 'BlockAssign_ST01_AL_SLDU.txt', 'BlockAssign_ST01_AL_CD.txt', 'BlockAssign_ST01_AL_VTD.txt', 'BlockAssign_ST01_AL_INCPLACE_CDP.txt', 'BlockAssign_ST01_AL_SLDL.txt', 'BlockAssign_ST01_AL_SDUNI.txt']
<class 'pandas.DataFrame'>
RangeIndex: 185976 entries, 0 to 185975
Data columns (total 2 columns):
 #   Column   Non-Null Count   Dtype
---  ------   --------------   -----
 0   BLOCKID  185976 non-null  str  
 1   PLACEFP  101328 non-null  str  
dtypes: str(2)
memory usage: 6.0 MB
None


,BLOCKID,PLACEFP
0,010010201001000,62328
1,010010201001001,62328
2,010010201001002,62328
3,010010201001003,NaN
4,010010201001004,NaN


In [4]:
# Examine the national mapper
parquet_path = Path("/home/eprashar_solutions_corelogic_com/network-idx/data/processed/census/baf2020/census_baf_national.parquet")
df = pd.read_parquet(parquet_path)
print(df.info())
print(df.head())

<class 'pandas.DataFrame'>
RangeIndex: 8174955 entries, 0 to 8174954
Data columns (total 5 columns):
 #   Column        Dtype
---  ------        -----
 0   block_geoid   str  
 1   state_fips    str  
 2   county_geoid  str  
 3   tract_geoid   str  
 4   place_geoid   str  
dtypes: str(5)
memory usage: 600.5 MB
None
       block_geoid state_fips county_geoid  tract_geoid place_geoid
0  020130001001000         02        02013  02013000100         NaN
1  020130001001001         02        02013  02013000100         NaN
2  020130001001002         02        02013  02013000100         NaN
3  020130001001003         02        02013  02013000100         NaN
4  020130001001004         02        02013  02013000100         NaN


In [5]:
# Count blocks where place_geoid is null
print(df["place_geoid"].isnull().sum())

3627401


In [6]:
# Checking how many places map to more than one unique county
place_county_counts = df.groupby("place_geoid")["county_geoid"].nunique()
print(place_county_counts.value_counts())

county_geoid
1    30607
2     1197
3       87
4       15
5        3
Name: count, dtype: int64


In [11]:
# For places that are tagged to more than one county; checking blocks per county
multi_county_places = place_county_counts[place_county_counts ==2].index
for place in multi_county_places[0:5]:  # Just checking the first 5 for now
    print(f"Place: {place}")
    print("="*85)
    counties = df[df["place_geoid"] == place]["county_geoid"].unique()
    for county in counties:
        block_count = df[(df["place_geoid"] == place) & (df["county_geoid"] == county)].shape[0]
        print(f"  County: {county} has {block_count} blocks")

Place: 0101660
  County: 01009 has 13 blocks
  County: 01055 has 56 blocks
Place: 0102116
  County: 01043 has 3 blocks
  County: 01095 has 342 blocks
Place: 0102320
  County: 01073 has 18 blocks
  County: 01115 has 84 blocks
Place: 0107000
  County: 01073 has 9107 blocks
  County: 01117 has 49 blocks
Place: 0107912
  County: 01055 has 57 blocks
  County: 01095 has 298 blocks
